# Creating a new tree

Goal: construct a phylogenetic tree of the EspY3 protein family across *Escherichia coli* and relatives (*Shigella* and *Escherichia albertii*. Include in the tree annotated *E. coli* tips with type information.

1. ## BLASTp

### Main BLAST search for tree backbone

Database: NCBI nr
Exluded WP_ to ensure strain-level metadata for *E. coli*
Filters:
- Taxonomy: Bacteria
- Max target sequences: 1000
Download: FASTA sequences > main.faa

This gives a broad *E. coli* representation, and includes Shigella and some unclassified enterobacteriaceae, with clean headers and strain information.

### Secondary search for *E. albertii*

A second search was necessary as *Albertii* sequences are distant and not represented in the above search, yet the protein appears to share a common ancestor.

Same search parameters except:
- Organism: *Escherichia albertii*
- Max targets: 200
Doanload: FASTA sequences > albertii_hits.faa

### Processing the data

Now that I have the fasta files, the next steps are to process the data.

1. Concatenate sequences
2. Reduce reduncancy (CD-HIT)
3. Multiple Sequence Allignment + Cleaning

In [7]:
# Concatenate queries

!cat raw/main_944.faa raw/albertii_hits.faa > raw/combined_raw.faa

2. ## CD-HIT to reduce reduncacy

The E. coli hits were all fairly high identity, so a CD-HIT to remove redunant/closely related copies is required.

However, I found in the BLASTp output that there are some *Shigella* hits with very high sequence identity is likely to get lost during the clustering process. Therefore, I will extract any sequences that aren't *E. coli* or *E. albertii* from the combined sequences, run CD-HIT, then rejoin them. 

CD-HIT at 98.5% identity gave 80 clusters which will be ideal for creating a tree once the shigella strains have been added back

In [13]:
# Make a file with everything EXCEPT Shigella (to run CD-HIT on)
!awk '/^>/{p = ($0 ~ /\[Shigella/)} p' raw/combined_raw.faa > raw/extracted_species/shigella.faa

# Make a file with ONLY Shigella (to add back after clustering)
!awk '/^>/{p = !($0 ~ /\[Shigella/)} p' raw/combined_raw.faa > raw/extracted_species/combined_no_shig_raw.faa

In [24]:
!cd-hit \
    -i raw/extracted_species/combined_no_shig_raw.fasta \
    -o cd-hit/combined_cd.fasta \
    -c 0.985

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 21:58:32
Command: cd-hit -i
         raw/extracted_species/combined_no_shig_raw.fasta -o
         cd-hit/combined_cd.fasta -c 0.985

Started: Fri Nov 14 19:25:43 2025
                            Output                              
----------------------------------------------------------------
total seq: 1079
longest and shortest : 543 and 146
Total letters: 552230
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 0M
Buffer          : 1 X 16M = 16M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 82M

Table limit with the given memory limit:
Max number of representatives: 1151167
Max number of word counting entries: 89725337

comparing sequences from          0  to       1079
.
     1079  finished         80  clusters

Approximated maximum memory consumption: 82M
writing new database
writing clustering information
program completed !

Total CPU time 0.09


3. ## Multiple Sequence Alignment and Cleaning

In [ ]:
# Add back the Shigella sequences
!cat cd-hit/combined_cd.fasta raw/extracted_species/shigella.fasta \
      > raw/final_for_alignment.faa

In [4]:
!mafft --maxiterate 1000 --localpair  raw/final_for_alignment.faa > mafft/alligned.faa

outputhat23=16
treein = 0
compacttree = 0
stacksize: 8176 kb
rescale = 1
All-to-all alignment.
tbfast-pair (aa) Version 7.526
alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

outputhat23=16
Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00
tbutree = 1, compacttree = 0
Constructing a UPGMA tree ... 
   90 / 92
done.

Progressive alignment ... 
STEP    66 /91 
Reallocating..done. *alloclen = 2088
STEP    91 /91 
done.
tbfast (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
1 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 16
sueff_global = 0.100000
nadd = 16
Loading 'hat3' ... done.
rescale = 1

   90 / 92
Segment   1/  1    1- 661
STEP 002-018-0  identical.   
Converged.

done
dvtditr (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strate

### Trimal

The allignment is good, but there are a number of empty columns created by insertions present in a few sequences, and in all of the E. albertii strains. 

Therefore, I will use trimal to remove gappy columns, but I will use the original MSA for mapping conservation onto structure.

In [7]:
!trimal -in mafft/alligned.fasta -out trimal/trimal.fasta -automated1

4. ## Tree Construction


5. ## Metadata Retrieval

6. ## Tree annotation